In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

DIR_M = 'season_splits/M/strength_model/'
DIR_W = 'season_splits/M/strength_model/'

In [35]:
m_train_str = pd.read_csv(DIR_M + 'train.csv')
w_train_str = pd.read_csv(DIR_W + 'train.csv')

In [36]:
w_train_str = w_train_str.drop(['Gender', 'GameType', 'isRegularSeason', 'isTourney', 'HasBoxscore','TotalScore','WLoc','WSeed', 'WSeedNum', 'WSeedRegion', 'LSeed','LSeedNum', 'LSeedRegion','NeutralSite','WCoachName', 'LCoachName','WMasseyMeanRank', 'WMasseyMedianRank', 'WMasseyBestRank','WMasseyWorstRank', 'LMasseyMeanRank', 'LMasseyMedianRank','LMasseyBestRank', 'LMasseyWorstRank','WCoachStartDayNum', 'WCoachEndDayNum','LCoachStartDayNum', 'LCoachEndDayNum','WMasseyTop10Count', 'WMasseyTop25Count', 'WMasseySystemCount','LMasseySystemCount','LMasseyTop10Count', 'LMasseyTop25Count','_row_order','_is_tournament'], axis=1)
m_train_str = m_train_str.drop(['Gender', 'GameType', 'isRegularSeason', 'isTourney', 'HasBoxscore','TotalScore','WLoc','WSeed', 'WSeedNum', 'WSeedRegion', 'LSeed','LSeedNum', 'LSeedRegion','NeutralSite','WCoachName', 'LCoachName','WMasseyMeanRank', 'WMasseyMedianRank', 'WMasseyBestRank','WMasseyWorstRank', 'LMasseyMeanRank', 'LMasseyMedianRank','LMasseyBestRank', 'LMasseyWorstRank','WCoachStartDayNum', 'WCoachEndDayNum','LCoachStartDayNum', 'LCoachEndDayNum','WMasseyTop10Count', 'WMasseyTop25Count', 'WMasseySystemCount','LMasseySystemCount','LMasseyTop10Count', 'LMasseyTop25Count','_row_order','_is_tournament'], axis=1)

In [38]:
w_train_str

,GameID,Season,DayNum,WTeamID,LTeamID,WScore,LScore,ScoreDiff,WHome,WAway,...,WMassey_ISR,WMassey_JCI,WMassey_LYD,WMassey_PAC,LMasseyDayNum,LMassey_DP,LMassey_ISR,LMassey_JCI,LMassey_LYD,LMassey_PAC
0,M_2003_10_1104_1328,2003,10,1104,1328,68,62,6,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M_2003_10_1272_1393,2003,10,1272,1393,70,63,7,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M_2003_11_1186_1458,2003,11,1458,1186,81,55,26,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M_2003_11_1208_1400,2003,11,1400,1208,77,71,6,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,M_2003_11_1266_1437,2003,11,1266,1437,73,61,12,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113236,M_2024_132_1120_1196,2024,132,1120,1196,86,67,19,0,0,...,NaN,NaN,NaN,4.0,128.0,33.0,NaN,NaN,NaN,25.0
113237,M_2024_132_1135_1463,2024,132,1463,1135,62,61,1,0,0,...,NaN,NaN,NaN,114.0,128.0,187.0,NaN,NaN,NaN,161.0
113238,M_2024_132_1182_1433,2024,132,1182,1433,57,51,6,0,0,...,NaN,NaN,NaN,78.0,128.0,86.0,NaN,NaN,NaN,93.0
113239,M_2024_132_1228_1458,2024,132,1228,1458,93,87,6,0,0,...,NaN,NaN,NaN,10.0,128.0,32.0,NaN,NaN,NaN,27.0


In [30]:
# =========================================================
# 1) HELPERY
# =========================================================

def safe_div(a, b):
    return np.where((b == 0) | pd.isna(b), np.nan, a / b)

def longest_streak(win_series):
    """
    Najdłuższa seria zwycięstw w serii bool/int (1=win, 0=loss).
    """
    arr = win_series.fillna(0).astype(int).to_numpy()
    best = cur = 0
    for x in arr:
        if x == 1:
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
    return best

def calc_elo_long(team_games, k=20, home_adv=75, reset_each_season=True):
    """
    Liczy Elo po kolei po meczach na danych team_games (1 wiersz = 1 drużyna w 1 meczu).
    Zakładamy, że dla jednego GameID są 2 wiersze: drużyna i przeciwnik.
    
    Zwraca:
      - team_games z kolumnami:
        EloPre, OppEloPre, EloPost
    """
    df = team_games.sort_values(["Season", "DayNum", "GameID", "TeamID"]).copy()

    ratings = {}
    out = []

    for season, g_season in df.groupby("Season", sort=True):
        if reset_each_season:
            ratings = {}

        # iterujemy po meczach
        for game_id, g_game in g_season.groupby("GameID", sort=False):
            if len(g_game) != 2:
                # dla bezpieczeństwa
                continue

            r1 = g_game.iloc[0]
            r2 = g_game.iloc[1]

            t1 = r1["TeamID"]
            t2 = r2["TeamID"]

            elo1 = ratings.get(t1, 1500.0)
            elo2 = ratings.get(t2, 1500.0)

            # home court
            adj1 = home_adv if r1["Home"] == 1 else (-home_adv if r1["Away"] == 1 else 0)
            adj2 = home_adv if r2["Home"] == 1 else (-home_adv if r2["Away"] == 1 else 0)

            exp1 = 1 / (1 + 10 ** (((elo2 + adj2) - (elo1 + adj1)) / 400))
            exp2 = 1 - exp1

            s1 = r1["Win"]
            s2 = r2["Win"]

            new_elo1 = elo1 + k * (s1 - exp1)
            new_elo2 = elo2 + k * (s2 - exp2)

            row1 = r1.to_dict()
            row1["EloPre"] = elo1
            row1["OppEloPre"] = elo2
            row1["EloPost"] = new_elo1

            row2 = r2.to_dict()
            row2["EloPre"] = elo2
            row2["OppEloPre"] = elo1
            row2["EloPost"] = new_elo2

            out.append(row1)
            out.append(row2)

            ratings[t1] = new_elo1
            ratings[t2] = new_elo2

    return pd.DataFrame(out)


# =========================================================
# 2) ZMIANA DO FORMATU TEAM-GAME
# =========================================================

def build_team_games(df):
    """
    Wejście: oryginalny df z kolumnami winner/loser.
    Wyjście: team_games = 1 wiersz = 1 drużyna w 1 meczu.
    """

    # --- zwycięzca jako perspektywa TeamID
    winners = pd.DataFrame({
        "GameID": df["GameID"],
        "Season": df["Season"],
        "DayNum": df["DayNum"],
        "TeamID": df["WTeamID"],
        "OppTeamID": df["LTeamID"],
        "PTS": df["WScore"],
        "PA": df["LScore"],
        "PointDiff": df["WScore"] - df["LScore"],
        "Win": 1,
        "Home": df["WHome"].astype(int),
        "Away": df["WAway"].astype(int),
        "Neutral": ((df["WHome"] == 0) & (df["WAway"] == 0)).astype(int),
        "NumOT": df["NumOT"],

        "FGM": df["WFGM"],
        "FGA": df["WFGA"],
        "FGM3": df["WFGM3"],
        "FGA3": df["WFGA3"],
        "FTM": df["WFTM"],
        "FTA": df["WFTA"],
        "OR": df["WOR"],
        "DR": df["WDR"],
        "Ast": df["WAst"],
        "TO": df["WTO"],
        "Stl": df["WStl"],
        "Blk": df["WBlk"],
        "PF": df["WPF"],

        "OppFGM": df["LFGM"],
        "OppFGA": df["LFGA"],
        "OppFGM3": df["LFGM3"],
        "OppFGA3": df["LFGA3"],
        "OppFTM": df["LFTM"],
        "OppFTA": df["LFTA"],
        "OppOR": df["LOR"],
        "OppDR": df["LDR"],
        "OppAst": df["LAst"],
        "OppTO": df["LTO"],
        "OppStl": df["LStl"],
        "OppBlk": df["LBlk"],
        "OppPF": df["LPF"],
    })

    # --- przegrany jako perspektywa TeamID
    losers = pd.DataFrame({
        "GameID": df["GameID"],
        "Season": df["Season"],
        "DayNum": df["DayNum"],
        "TeamID": df["LTeamID"],
        "OppTeamID": df["WTeamID"],
        "PTS": df["LScore"],
        "PA": df["WScore"],
        "PointDiff": df["LScore"] - df["WScore"],
        "Win": 0,
        "Home": df["LHome"].astype(int),
        "Away": df["LAway"].astype(int),
        "Neutral": ((df["LHome"] == 0) & (df["LAway"] == 0)).astype(int),
        "NumOT": df["NumOT"],

        "FGM": df["LFGM"],
        "FGA": df["LFGA"],
        "FGM3": df["LFGM3"],
        "FGA3": df["LFGA3"],
        "FTM": df["LFTM"],
        "FTA": df["LFTA"],
        "OR": df["LOR"],
        "DR": df["LDR"],
        "Ast": df["LAst"],
        "TO": df["LTO"],
        "Stl": df["LStl"],
        "Blk": df["LBlk"],
        "PF": df["LPF"],

        "OppFGM": df["WFGM"],
        "OppFGA": df["WFGA"],
        "OppFGM3": df["WFGM3"],
        "OppFGA3": df["WFGA3"],
        "OppFTM": df["WFTM"],
        "OppFTA": df["WFTA"],
        "OppOR": df["WOR"],
        "OppDR": df["WDR"],
        "OppAst": df["WAst"],
        "OppTO": df["WTO"],
        "OppStl": df["WStl"],
        "OppBlk": df["WBlk"],
        "OppPF": df["WPF"],
    })

    team_games = pd.concat([winners, losers], ignore_index=True)
    team_games = team_games.sort_values(["Season", "TeamID", "DayNum", "GameID"]).reset_index(drop=True)

    # possessions
    team_games["Poss"] = (
        team_games["FGA"] + 0.44 * team_games["FTA"] - team_games["OR"] + team_games["TO"]
    )
    team_games["OppPoss"] = (
        team_games["OppFGA"] + 0.44 * team_games["OppFTA"] - team_games["OppOR"] + team_games["OppTO"]
    )

    # meczowe efficiency / rate'y
    team_games["OffEff"] = safe_div(team_games["PTS"], team_games["Poss"])
    team_games["DefEff"] = safe_div(team_games["PA"], team_games["OppPoss"])
    team_games["NetRating"] = team_games["OffEff"] - team_games["DefEff"]

    team_games["eFG"] = safe_div(team_games["FGM"] + 0.5 * team_games["FGM3"], team_games["FGA"])
    team_games["TS"] = safe_div(team_games["PTS"], 2 * (team_games["FGA"] + 0.44 * team_games["FTA"]))
    team_games["ThreePAR"] = safe_div(team_games["FGA3"], team_games["FGA"])
    team_games["BlockRate"] = safe_div(team_games["Blk"], team_games["OppFGA"])
    team_games["StealRate"] = safe_div(team_games["Stl"], team_games["OppPoss"])
    team_games["AssistRate"] = safe_div(team_games["Ast"], team_games["FGM"])
    team_games["FTRate"] = safe_div(team_games["FTA"], team_games["FGA"])
    team_games["TOPct"] = safe_div(team_games["TO"], team_games["Poss"])
    team_games["ORPct"] = safe_div(team_games["OR"], team_games["OR"] + team_games["OppDR"])
    team_games["DRPct"] = safe_div(team_games["DR"], team_games["DR"] + team_games["OppOR"])
    team_games["ReboundPct"] = safe_div(team_games["OR"] + team_games["DR"],
                                        team_games["OR"] + team_games["DR"] + team_games["OppOR"] + team_games["OppDR"])
    team_games["FoulRate"] = safe_div(team_games["PF"], team_games["OppPoss"])

    team_games["CloseGame"] = (team_games["PointDiff"].abs() <= 5).astype(int)
    team_games["CloseWin"] = ((team_games["PointDiff"] > 0) & (team_games["PointDiff"].abs() <= 5)).astype(int)
    team_games["Blowout"] = (team_games["PointDiff"].abs() >= 15).astype(int)
    team_games["OvertimeGame"] = (team_games["NumOT"] > 0).astype(int)

    return team_games


# =========================================================
# 3) RECENT / ROLLING FEATURES
# =========================================================

def add_recent_features(team_games, windows=(5, 10)):
    """
    Dodaje rolling statystyki liczone po ostatnich N meczach
    wewnątrz (Season, TeamID).
    """
    df = team_games.sort_values(["Season", "TeamID", "DayNum", "GameID"]).copy()
    grp = df.groupby(["Season", "TeamID"], group_keys=False)

    for w in windows:
        # win%
        df[f"win_pct_last{w}"] = grp["Win"].transform(
            lambda s: s.rolling(w, min_periods=1).mean()
        )

        # point diff
        df[f"point_diff_last{w}"] = grp["PointDiff"].transform(
            lambda s: s.rolling(w, min_periods=1).mean()
        )

        # recent off / def rating - lepiej z sum niż ze średniej z ratio
        df[f"pts_last{w}"] = grp["PTS"].transform(lambda s: s.rolling(w, min_periods=1).sum())
        df[f"pa_last{w}"] = grp["PA"].transform(lambda s: s.rolling(w, min_periods=1).sum())
        df[f"poss_last{w}"] = grp["Poss"].transform(lambda s: s.rolling(w, min_periods=1).sum())
        df[f"opp_poss_last{w}"] = grp["OppPoss"].transform(lambda s: s.rolling(w, min_periods=1).sum())

        df[f"off_rating_last{w}"] = safe_div(df[f"pts_last{w}"], df[f"poss_last{w}"])
        df[f"def_rating_last{w}"] = safe_div(df[f"pa_last{w}"], df[f"opp_poss_last{w}"])
        df[f"net_rating_last{w}"] = df[f"off_rating_last{w}"] - df[f"def_rating_last{w}"]

        # recent eFG%
        df[f"fgm_last{w}"] = grp["FGM"].transform(lambda s: s.rolling(w, min_periods=1).sum())
        df[f"fgm3_last{w}"] = grp["FGM3"].transform(lambda s: s.rolling(w, min_periods=1).sum())
        df[f"fga_last{w}"] = grp["FGA"].transform(lambda s: s.rolling(w, min_periods=1).sum())
        df[f"eFG_last{w}"] = safe_div(df[f"fgm_last{w}"] + 0.5 * df[f"fgm3_last{w}"], df[f"fga_last{w}"])

        # recent turnover%
        df[f"to_last{w}"] = grp["TO"].transform(lambda s: s.rolling(w, min_periods=1).sum())
        df[f"topct_last{w}"] = safe_div(df[f"to_last{w}"], df[f"poss_last{w}"])

        # recent rebound%
        df[f"or_last{w}"] = grp["OR"].transform(lambda s: s.rolling(w, min_periods=1).sum())
        df[f"dr_last{w}"] = grp["DR"].transform(lambda s: s.rolling(w, min_periods=1).sum())
        df[f"oppor_last{w}"] = grp["OppOR"].transform(lambda s: s.rolling(w, min_periods=1).sum())
        df[f"oppdr_last{w}"] = grp["OppDR"].transform(lambda s: s.rolling(w, min_periods=1).sum())

        df[f"rebound_pct_last{w}"] = safe_div(
            df[f"or_last{w}"] + df[f"dr_last{w}"],
            df[f"or_last{w}"] + df[f"dr_last{w}"] + df[f"oppor_last{w}"] + df[f"oppdr_last{w}"]
        )

    return df


# =========================================================
# 4) AGREGACJE SEZONOWE
# =========================================================

def build_season_team_features(df_raw):
    # --- 1. long format
    team_games = build_team_games(df_raw)

    # --- 2. Elo
    team_games = calc_elo_long(team_games, k=20, home_adv=75, reset_each_season=True)

    # --- 3. recent features
    team_games = add_recent_features(team_games, windows=(5, 10))

    # --- 4. agregacje sezonowe
    base = team_games.groupby(["Season", "TeamID"]).agg(
        total_games=("GameID", "count"),
        wins=("Win", "sum"),

        win_percentage=("Win", "mean"),
        avg_points_diff=("PointDiff", "mean"),
        avg_points_scored=("PTS", "mean"),
        avg_points_allowed=("PA", "mean"),
        median_points_diff=("PointDiff", "median"),

        close_games=("CloseGame", "sum"),
        close_game_wins=("CloseWin", "sum"),
        blowout_rate=("Blowout", "mean"),
        overtime_rate=("OvertimeGame", "mean"),

        total_pts=("PTS", "sum"),
        total_pa=("PA", "sum"),
        total_poss=("Poss", "sum"),
        total_opp_poss=("OppPoss", "sum"),

        total_fgm=("FGM", "sum"),
        total_fga=("FGA", "sum"),
        total_fgm3=("FGM3", "sum"),
        total_fga3=("FGA3", "sum"),
        total_fta=("FTA", "sum"),
        total_ast=("Ast", "sum"),
        total_to=("TO", "sum"),
        total_or=("OR", "sum"),
        total_dr=("DR", "sum"),
        total_blk=("Blk", "sum"),
        total_stl=("Stl", "sum"),
        total_pf=("PF", "sum"),

        total_opp_fga=("OppFGA", "sum"),
        total_opp_dr=("OppDR", "sum"),
        total_opp_or=("OppOR", "sum"),

        home_games=("Home", "sum"),
        away_games=("Away", "sum"),
        neutral_games=("Neutral", "sum"),

        home_wins=("Win", lambda s: s[team_games.loc[s.index, "Home"] == 1].sum()),
        away_wins=("Win", lambda s: s[team_games.loc[s.index, "Away"] == 1].sum()),
        neutral_wins=("Win", lambda s: s[team_games.loc[s.index, "Neutral"] == 1].sum()),

        home_point_diff=("PointDiff", lambda s: s[team_games.loc[s.index, "Home"] == 1].mean()),
        away_point_diff=("PointDiff", lambda s: s[team_games.loc[s.index, "Away"] == 1].mean()),
        neutral_point_diff=("PointDiff", lambda s: s[team_games.loc[s.index, "Neutral"] == 1].mean()),
    ).reset_index()

    # close game winrate
    base["close_game_winrate"] = safe_div(base["close_game_wins"], base["close_games"])

    # efficiency / advanced
    base["offensive_eff"] = safe_div(base["total_pts"], base["total_poss"])
    base["defensive_eff"] = safe_div(base["total_pa"], base["total_opp_poss"])
    base["net_rating"] = base["offensive_eff"] - base["defensive_eff"]

    base["possessions"] = safe_div(base["total_poss"], base["total_games"])  # średnie poss per game
    base["tempo"] = safe_div(base["total_poss"], base["total_games"])        # wg Twojej definicji to samo

    base["eff_field_goal_percentage"] = safe_div(
        base["total_fgm"] + 0.5 * base["total_fgm3"], base["total_fga"]
    )
    base["true_shooting_percentage"] = safe_div(
        base["total_pts"], 2 * (base["total_fga"] + 0.44 * base["total_fta"])
    )
    base["three_points_attempt_rate"] = safe_div(base["total_fga3"], base["total_fga"])
    base["block_rate"] = safe_div(base["total_blk"], base["total_opp_fga"])
    base["steal_rate"] = safe_div(base["total_stl"], base["total_opp_poss"])
    base["assist_rate"] = safe_div(base["total_ast"], base["total_fgm"])
    base["free_throws_rate"] = safe_div(base["total_fta"], base["total_fga"])
    base["turnover_percentage"] = safe_div(base["total_to"], base["total_poss"])
    base["offensive_rebound_percentage"] = safe_div(
        base["total_or"], base["total_or"] + base["total_opp_dr"]
    )
    base["def_reb_perc"] = safe_div(
        base["total_dr"], base["total_dr"] + base["total_opp_or"]
    )
    base["foul_rate"] = safe_div(base["total_pf"], base["total_opp_poss"])

    # location split win%
    base["home_win_pct"] = safe_div(base["home_wins"], base["home_games"])
    base["away_win_pct"] = safe_div(base["away_wins"], base["away_games"])
    base["neutral_win_pct"] = safe_div(base["neutral_wins"], base["neutral_games"])

    # home/away split deltas
    base["home_away_win_pct_delta"] = base["home_win_pct"] - base["away_win_pct"]
    base["home_away_point_diff_delta"] = base["home_point_diff"] - base["away_point_diff"]

    # longest win streak
    streaks = (
        team_games
        .sort_values(["Season", "TeamID", "DayNum", "GameID"])
        .groupby(["Season", "TeamID"])["Win"]
        .apply(longest_streak)
        .reset_index(name="longest_win_streak")
    )

    base = base.merge(streaks, on=["Season", "TeamID"], how="left")

    # --- 5. final recent features = bierzemy ostatni mecz w sezonie
    recent_cols = [
        "Season", "TeamID", "EloPost",
        "win_pct_last5", "win_pct_last10",
        "point_diff_last5", "point_diff_last10",
        "off_rating_last5", "off_rating_last10",
        "def_rating_last5", "def_rating_last10",
        "net_rating_last5", "net_rating_last10",
        "eFG_last5", "eFG_last10",
        "topct_last5", "topct_last10",
        "rebound_pct_last5", "rebound_pct_last10",
    ]

    recent = (
        team_games
        .sort_values(["Season", "TeamID", "DayNum", "GameID"])
        .groupby(["Season", "TeamID"])
        .tail(1)[recent_cols]
        .rename(columns={"EloPost": "recent_elo"})
    )

    base = base.merge(recent, on=["Season", "TeamID"], how="left")

    # trend
    base["trend_last5"] = base["net_rating_last5"] - base["net_rating"]
    base["trend_last10"] = base["net_rating_last10"] - base["net_rating"]

    # porządek kolumn
    cols_first = [
        "Season", "TeamID", "total_games", "wins",
        "win_percentage", "avg_points_diff", "avg_points_scored", "avg_points_allowed",
        "median_points_diff", "close_game_winrate", "blowout_rate", "overtime_rate",
        "longest_win_streak",
        "offensive_eff", "defensive_eff", "net_rating", "possessions", "tempo",
        "eff_field_goal_percentage", "true_shooting_percentage",
        "three_points_attempt_rate", "block_rate", "steal_rate", "assist_rate",
        "free_throws_rate", "turnover_percentage",
        "offensive_rebound_percentage", "def_reb_perc", "foul_rate",
        "home_win_pct", "away_win_pct", "neutral_win_pct",
        "home_point_diff", "away_point_diff", "neutral_point_diff",
        "home_away_win_pct_delta", "home_away_point_diff_delta",
        "win_pct_last5", "win_pct_last10",
        "point_diff_last5", "point_diff_last10",
        "off_rating_last5", "off_rating_last10",
        "def_rating_last5", "def_rating_last10",
        "eFG_last5", "eFG_last10",
        "topct_last5", "topct_last10",
        "rebound_pct_last5", "rebound_pct_last10",
        "recent_elo", "trend_last5", "trend_last10"
    ]

    cols_first = [c for c in cols_first if c in base.columns] + [c for c in base.columns if c not in cols_first]
    base = base[cols_first]

    return base, team_games

In [ ]:
season_features_m, team_games_m = build_season_team_features(m_train_str)


In [43]:
season_features_w, team_games_w = build_season_team_features(w_train_str)


In [45]:
pd.set_option("display.max_rows", None)

# pokaż wszystkie kolumny
pd.set_option("display.max_columns", None)


null_table_m = (
    season_features_m.isnull().sum()
    .to_frame(name="null_count")
)

null_table_m["null_percent"] = (
    null_table_m["null_count"] / len(season_features_m) * 100
)

print(null_table_m)

                              null_count  null_percent
Season                                 0           0.0
TeamID                                 0           0.0
total_games                            0           0.0
wins                                   0           0.0
win_percentage                         0           0.0
avg_points_diff                        0           0.0
avg_points_scored                      0           0.0
avg_points_allowed                     0           0.0
median_points_diff                     0           0.0
close_game_winrate                     0           0.0
blowout_rate                           0           0.0
overtime_rate                          0           0.0
longest_win_streak                     0           0.0
offensive_eff                          0           0.0
defensive_eff                          0           0.0
net_rating                             0           0.0
possessions                            0           0.0
tempo     

In [41]:
def fix_neutral(season_features):

    season_features = season_features.fillna(0)
    k = 5
    season_features["neutral_point_diff_smothered"] = (
    (season_features["neutral_games"] / (season_features["neutral_games"] + k)) * season_features["neutral_point_diff"] +
    (k / (season_features["neutral_games"] + k)) * (
        season_features["neutral_point_diff"] +
        season_features["home_point_diff"] +
        season_features["away_point_diff"]
    )
    )
    season_features["neutral_win_pct_smothered"] = (
        (season_features["neutral_games"] / (season_features["neutral_games"] + k)) * season_features["neutral_win_pct"] +
        (k / (season_features["neutral_games"] + k)) * (
            season_features["neutral_win_pct"] +
            season_features["home_point_diff"] +
            season_features["away_point_diff"]
        )
    )
    return season_features

In [44]:
season_features_m = fix_neutral(season_features_m)
season_features_w = fix_neutral(season_features_w)

In [ ]:
season_features_m.to_csv("season_features_m.csv", index=False)
season_features_w.to_csv("season_features_w.csv", index=False)